# TritFabric Quantum Lane — Demo

This demo does two things:
1. **SU(2) charge estimation** (Pauli expectation) using Qiskit Estimator (Aer by default; IBM Runtime if enabled).
2. **QAOA routing toy** on a 64-node heavy-hex-like subgraph to pick a schedule (MaxCut proxy).

Results are saved under `../out/` and a Markdown report is rendered + signed.

In [1]:
import os, json, math, time, pathlib
from quantum.adapter import BackendConfig, QuantumChargeProjector, QAOAScheduler
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
out_dir = pathlib.Path('../out'); out_dir.mkdir(parents=True, exist_ok=True)

# Backend selection via env:
provider = 'ibm' if os.environ.get('TF_QUANTUM_PROVIDER')=='ibm' else 'aer'
cfg = BackendConfig(provider=provider, backend=os.environ.get('TF_BACKEND','heron'), shots=int(os.environ.get('TF_SHOTS','2000')))
projector = QuantumChargeProjector(cfg)
cfg.__dict__

## Part 1 — SU(2) charge estimate

In [2]:
# Build a small 3-qubit circuit and an observable J = a1*Z0 + a2*Z1Z2 + a3*X0X1
a1, a2, a3 = 0.7, -0.4, 0.5
qc = QuantumCircuit(3)
qc.ry(0.83,0); qc.rx(0.2,1); qc.ry(-0.5,2)
qc.cx(0,1); qc.cz(1,2)

op = SparsePauliOp.from_list([
    ('ZII', a1),
    ('IZZ', a2),
    ('XXI', a3)
])
mean, var = projector.estimate(qc, op)
su2_result = {'mean': float(mean), 'variance': float(var)}
su2_result

## Part 2 — QAOA schedule on heavy-hex-like subgraph (n=64, p=2)

In [3]:
sched = QAOAScheduler(cfg)
best = sched.run_grid_search(n=64, p_layers=2, gamma_grid=(-1.5,-0.5,0.5,1.5), beta_grid=(0.2,0.6,1.0))
best

## Save artifacts and publish report

In [4]:
import datetime, hashlib, json, os
ts = datetime.datetime.utcnow().isoformat()+"Z"
provenance = {
  'timestamp': ts,
  'backend': cfg.__dict__,
  'su2': su2_result,
  'qaoa_best': best
}
(out_dir/'provenance.json').write_text(json.dumps(provenance, indent=2))

md = f"""
# Quantum Lane Proof Report

**Timestamp (UTC):** {ts}

## Backend
```json
{json.dumps(cfg.__dict__, indent=2)}
```

## SU(2) Charge (Estimator)
```json
{json.dumps(su2_result, indent=2)}
```

## QAOA Best (n=64, p=2)
```json
{json.dumps(best, indent=2)}
```
“Proof-of-work” = this JSON + signed PDF stored in Shifu.
"""
(out_dir/'report.md').write_text(md)
print('Artifacts written to', out_dir)